<a href="https://colab.research.google.com/github/anishkumarcs24/AI/blob/main/1BF24CS053_Week8_FOL_Resolution/1BF24CS053_Week8_FOL_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# FIRST ORDER LOGIC RESOLUTION
# Case Study: Prove that John likes peanuts.

# ------------------------------------------------
# 1. REPRESENTATION OF KNOWLEDGE BASE
# ------------------------------------------------

# A literal is represented as:
# (negation, predicate, arguments)
# True = Negated, False = Positive
#
# Variables start with '?'
# Constants start with capital letters.

C1 = [(True, "Food", ("?x",)),
      (False, "Likes", ("John", "?x"))]

C2 = [(False, "Food", ("Apple",))]

C3 = [(False, "Food", ("Vegetables",))]

C4 = [(True, "Eats", ("?x", "?y")),
      (False, "Killed", ("?x",)),
      (False, "Food", ("?y",))]

C5 = [(False, "Eats", ("Anil", "Peanuts"))]

C6 = [(False, "Alive", ("Anil",))]

C7 = [(True, "Eats", ("Anil", "?x")),
      (False, "Eats", ("Harry", "?x"))]

C8 = [(True, "Alive", ("?x",)),
      (True, "Killed", ("?x",))]

C9 = [(False, "Killed", ("?x",)),
      (False, "Alive", ("?x",))]

# Negation of the query:
# Query = Likes(John, Peanuts)

C10 = [(True, "Likes", ("John", "Peanuts"))]

KB = [C1, C2, C3, C4, C5,
      C6, C7, C8, C9, C10]


# ------------------------------------------------
# 2. DISPLAY CLAUSES
# ------------------------------------------------

def format_literal(literal):
    neg, predicate, args = literal

    text = predicate + "(" + ", ".join(args) + ")"

    if neg:
        return "~" + text

    return text


def format_clause(clause):
    if not clause:
        return "EMPTY CLAUSE (□)"

    return " OR ".join(
        format_literal(lit) for lit in clause
    )


print("KNOWLEDGE BASE IN CNF\n")

for i, clause in enumerate(KB, 1):
    print("C" + str(i) + ":", format_clause(clause))


# ------------------------------------------------
# 3. UNIFICATION
# ------------------------------------------------

def is_variable(term):
    return term.startswith("?")


def substitute(term, substitution):

    while is_variable(term) and term in substitution:
        term = substitution[term]

    return term


def unify(term1, term2, substitution):

    term1 = substitute(term1, substitution)
    term2 = substitute(term2, substitution)

    if term1 == term2:
        return True

    if is_variable(term1):
        substitution[term1] = term2
        return True

    if is_variable(term2):
        substitution[term2] = term1
        return True

    return False


# ------------------------------------------------
# 4. APPLY SUBSTITUTION
# ------------------------------------------------

def apply_substitution(literal, substitution):

    neg, predicate, args = literal

    new_args = tuple(
        substitute(arg, substitution) for arg in args
    )

    return (neg, predicate, new_args)


# ------------------------------------------------
# 5. RESOLUTION
# ------------------------------------------------

def resolve(clause1, clause2):

    for i, lit1 in enumerate(clause1):

        for j, lit2 in enumerate(clause2):

            neg1, pred1, args1 = lit1
            neg2, pred2, args2 = lit2

            # Check complementary literals
            if pred1 != pred2 or neg1 == neg2:
                continue

            if len(args1) != len(args2):
                continue

            substitution = {}
            success = True

            # Perform unification
            for a, b in zip(args1, args2):

                if not unify(a, b, substitution):
                    success = False
                    break

            if success:

                # Remove resolved literals
                remaining = (
                    clause1[:i] + clause1[i+1:] +
                    clause2[:j] + clause2[j+1:]
                )

                # Apply substitution
                result = [
                    apply_substitution(lit, substitution)
                    for lit in remaining
                ]

                # Remove duplicate literals
                result = list(dict.fromkeys(result))

                return result

    return None


# ------------------------------------------------
# 6. PERFORM RESOLUTION FOR THE CASE STUDY
# ------------------------------------------------

print("\nRESOLUTION PROCESS\n")

proof = []

def perform_resolution(id1, id2, clause1, clause2, new_id):

    result = resolve(clause1, clause2)

    print("Resolving C" + str(id1) +
          " and C" + str(id2))

    if result is None:
        print("Resolution failed!")
        return None

    print("C" + str(new_id) + ":",
          format_clause(result))

    print()

    proof.append(
        (new_id, id1, id2, result)
    )

    return result


# Step 1: Resolve C6 and C8
# Alive(Anil) and ~Alive(x) OR ~Killed(x)

C11 = perform_resolution(6, 8, C6, C8, 11)


# Step 2: Resolve C4 and C5
# Anything eaten and not killed is food
# Anil eats peanuts

C12 = perform_resolution(4, 5, C4, C5, 12)


# Step 3: Resolve C12 and C11
# Killed(Anil) OR Food(Peanuts)
# ~Killed(Anil)

C13 = perform_resolution(12, 11, C12, C11, 13)


# Step 4: Resolve C1 and C13
# John likes all food
# Peanuts are food

C14 = perform_resolution(1, 13, C1, C13, 14)


# Step 5: Resolve C14 and C10
# Likes(John, Peanuts)
# ~Likes(John, Peanuts)

C15 = perform_resolution(14, 10, C14, C10, 15)


# ------------------------------------------------
# 7. DISPLAY FINAL RESULT
# ------------------------------------------------

print("FINAL RESULT")

if C15 == []:

    print("Empty clause generated: □")
    print("Contradiction obtained.")
    print("QUERY PROVED SUCCESSFULLY!")
    print("\nJohn likes peanuts.")

else:

    print("Query could not be proved.")


# ------------------------------------------------
# 8. DISPLAY RESOLUTION PROOF TREE
# ------------------------------------------------

print("\nRESOLUTION PROOF TREE\n")

print("""
                    C15 (□)
                   / \\
                C14 C10
                 |
        Likes(John, Peanuts)
                 |
                / \\
              C1 C13
                   |
             Food(Peanuts)
                   |
                  / \\
                C12 C11
                 | |
                 | ~Killed(Anil)
                 |
       Killed(Anil) OR Food(Peanuts)
                 |
                / \\
              C4 C5
""")

print("Derivation of C11:")
print("C6 + C8 = ~Killed(Anil)")

KNOWLEDGE BASE IN CNF

C1: ~Food(?x) OR Likes(John, ?x)
C2: Food(Apple)
C3: Food(Vegetables)
C4: ~Eats(?x, ?y) OR Killed(?x) OR Food(?y)
C5: Eats(Anil, Peanuts)
C6: Alive(Anil)
C7: ~Eats(Anil, ?x) OR Eats(Harry, ?x)
C8: ~Alive(?x) OR ~Killed(?x)
C9: Killed(?x) OR Alive(?x)
C10: ~Likes(John, Peanuts)

RESOLUTION PROCESS

Resolving C6 and C8
C11: ~Killed(Anil)

Resolving C4 and C5
C12: Killed(Anil) OR Food(Peanuts)

Resolving C12 and C11
C13: Food(Peanuts)

Resolving C1 and C13
C14: Likes(John, Peanuts)

Resolving C14 and C10
C15: EMPTY CLAUSE (□)

FINAL RESULT
Empty clause generated: □
Contradiction obtained.
QUERY PROVED SUCCESSFULLY!

John likes peanuts.

RESOLUTION PROOF TREE


                    C15 (□)
                   / \
                C14 C10
                 |
        Likes(John, Peanuts)
                 |
                / \
              C1 C13
                   |
             Food(Peanuts)
                   |
                  / \
                C12 C11
             